In [ ]:
from astropy.table import Table
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import pymc3 as pm
import pymc3_ext as pmx
import exoplanet as xo
import os
from corner import corner
import theano.tensor as tt
from tqdm.notebook import tqdm
from scipy.stats import median_abs_deviation
import pandas as pd
from astroquery.mast import Catalogs
from scipy.interpolate import UnivariateSpline
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
# ============================================================
# Directorio para guardar figuras
# ============================================================

output_dir = Path(
    "/home/luciam/Documents/starry/Resultados_Orion/age_distributions"
)

output_dir.mkdir(parents=True, exist_ok=True)



In [ ]:
# Catalogo de resultados obtenidos por massage

path_cat = "/home/luciam/Documents/starry/Resultados_Orion/catalogo_completo_mist_sommers.csv"

cat = pd.read_csv(path_cat)
cat.columns




In [ ]:
#print(cat.columns)
## DATOS MIST


#variables de datos obtenidos por massage
# ============================================================
temp_mist = cat["T_mean_1"]
temp_std_mist = cat["T_std_1"]
temp_median_mist = cat['T_median_1']
# ============================================================
lgL_mean_mist = cat['lgL_mean_1']
lgL_std_mist = cat['lgL_std_1']
lgL_median_mist = cat["lgL_median_1"]
# ============================================================
edad_mean_mist = cat["age_mean_1"]
edad_std_mist = cat["age_std_1"]
edad_median_mist = cat["age_median_1"]
TICs = cat['TIC']

# ============================================================

T_sun = 5772.0  # K

R_mean_mist = (
    10**(lgL_mean_mist / 2)
    * (T_sun / temp_mist)**2
)

R_median_mist = (
    10**(lgL_median_mist / 2)
    * (T_sun / temp_median_mist)**2
)
# ============================================================
# Resumen básico
# ============================================================

edad_summary = pd.DataFrame({
    "TIC": TICs,
    "age_mean": edad_mean_mist,
    "age_std": edad_std_mist,
    "age_median": edad_median_mist
})

#print(edad_summary)
#cat.head

print(edad_summary.describe())

print("\nValores nulos:")
print(edad_summary.isna().sum())

print("\nEdades únicas:")
print(np.sort(edad_summary["age_median"].dropna().unique()))

In [ ]:

# ============================================================
# Graficar distribución de edad para cada TIC
# ============================================================

for _, row in edad_summary.iterrows():

    tic = row["TIC"]
    age_mean = row["age_mean"]
    age_std = row["age_std"]
    age_median = row["age_median"]

    # Saltar valores inválidos
    if pd.isna(age_mean) or pd.isna(age_std):
        continue

    if age_std <= 0:
        continue

    # --------------------------------------------------------
    # Rango de edades
    # --------------------------------------------------------

    age_min = max(0, age_mean - 4 * age_std)
    age_max = age_mean + 4 * age_std

    ages = np.linspace(age_min, age_max, 1000)

    # Distribución normal
    pdf = (
        1 / (age_std * np.sqrt(2 * np.pi))
        * np.exp(-0.5 * ((ages - age_mean) / age_std) ** 2)
    )

    # --------------------------------------------------------
    # Figura
    # --------------------------------------------------------

    fig, ax = plt.subplots(figsize=(8, 5))

    ax.plot(
        ages,
        pdf,
        linewidth=2
    )

    # Media
    ax.axvline(
        age_mean,
        linestyle="--",
        linewidth=1.8,
        label=f"Mean = {age_mean:.2f}"
    )

    # Mediana
    if not pd.isna(age_median):

        ax.axvline(
            age_median,
            linestyle=":",
            linewidth=2,
            label=f"Median = {age_median:.2f}"
        )

    # Región ±1 sigma
    ax.axvspan(
        age_mean - age_std,
        age_mean + age_std,
        alpha=0.15,
        label=f"$1\\sigma$ = {age_std:.2f}"
    )

    # --------------------------------------------------------
    # Labels
    # --------------------------------------------------------

    ax.set_xlabel("Age", fontsize=13)
    ax.set_ylabel("Probability density", fontsize=13)

    ax.set_title(
        f"TIC {tic}\n"
        f"Age = {age_mean:.2f} ± {age_std:.2f} Myr (MIST)",
        fontsize=14
    )

    ax.legend()

    ax.grid(
        alpha=0.25,
        linestyle="--"
    )

    plt.tight_layout()

    # --------------------------------------------------------
    # Guardar
    # --------------------------------------------------------

    filename = output_dir / f"TIC_{tic}_age_distribution_mist.png"

    fig.savefig(
        filename,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

In [ ]:

### EDADES MIST

edad_summary["age_relative_error"] = (
    edad_summary["age_std"] /
    edad_summary["age_mean"]
)

edad_summary["age_lower"] = (
    edad_summary["age_mean"] -
    edad_summary["age_std"]
).clip(lower=0)

edad_summary["age_upper"] = (
    edad_summary["age_mean"] +
    edad_summary["age_std"]
)

edad_summary["age_lower_2sigma"] = (
    edad_summary["age_mean"] -
    2 * edad_summary["age_std"]
).clip(lower=0)

edad_summary["age_upper_2sigma"] = (
    edad_summary["age_mean"] +
    2 * edad_summary["age_std"]
)

display(
    edad_summary.sort_values("age_relative_error")
)

def classify_age_uncertainty(relative_error):

    if pd.isna(relative_error):
        return "No disponible"

    elif relative_error < 0.20:
        return "Muy buena"

    elif relative_error < 0.40:
        return "Buena"

    elif relative_error < 0.60:
        return "Moderada"

    elif relative_error < 1.00:
        return "Pobre"

    else:
        return "Muy pobre"


edad_summary["age_quality"] = (
    edad_summary["age_relative_error"]
    .apply(classify_age_uncertainty)
)

display(
    edad_summary[
        [
            "TIC",
            "age_mean",
            "age_std",
            "age_median",
            "age_relative_error",
            "age_quality"
        ]
    ]
)
print(
    edad_summary["age_quality"].value_counts()
)

fig, ax = plt.subplots(figsize=(10, 7))

ax.errorbar(
    edad_summary["age_mean"],
    np.arange(len(edad_summary)),
    xerr=edad_summary["age_std"],
    fmt="o",
    capsize=4
)

ax.set_yticks(np.arange(len(edad_summary)))
ax.set_yticklabels(
    [f"TIC {tic}" for tic in edad_summary["TIC"]]
)

ax.set_xlabel("Age", fontsize=13)
ax.set_ylabel("TIC", fontsize=13)

ax.set_title(
    "Age estimates for the Orion TIC sample mist",
    fontsize=15
)

ax.grid(
    axis="x",
    alpha=0.25,
    linestyle="--"
)

plt.tight_layout()
plt.show()

# Edades modelo de somers 17


In [ ]:
### Datos somers 17 

#resultados massage
# ============================================================
temp_s = cat["T_mean_2"]
temp_std_s = cat["T_std_2"]
temp_median_s = cat['T_median_2']
# ============================================================
lgL_mean_s = cat['lgL_mean_2']
lgL_std_s = cat['lgL_std_2']
lgL_median_s = cat["lgL_median_2"]
# ============================================================
edad_mean_s = cat["age_mean_2"]
edad_std_s = cat["age_std_2"]
edad_median_s = cat["age_median_2"]


# ============================================================


T_sun = 5772.0  # K

R_mean_s = (
    10**(lgL_mean_s / 2)
    * (T_sun / temp_s)**2
)

R_median_s = (
    10**(lgL_median_s / 2)
    * (T_sun / temp_median_s)**2
)

# ============================================================
# Resumen básico
# ============================================================

edad_summary2 = pd.DataFrame({
    "TIC": TICs,
    "age_mean": edad_mean_s,
    "age_std": edad_std_s,
    "age_median": edad_median_s
})

#print(edad_summary)
#cat.head

print(edad_summary2.describe())

print("\nValores nulos:")
print(edad_summary2.isna().sum())

print("\nEdades únicas:")
print(np.sort(edad_summary2["age_median"].dropna().unique()))

In [ ]:

#EDADES somers 

edad_summary2["age_relative_error"] = (
    edad_summary2["age_std"] /
    edad_summary2["age_mean"]
)

edad_summary2["age_lower"] = (
    edad_summary2["age_mean"] -
    edad_summary2["age_std"]
).clip(lower=0)

edad_summary2["age_upper"] = (
    edad_summary2["age_mean"] +
    edad_summary2["age_std"]
)

edad_summary2["age_lower_2sigma"] = (
    edad_summary2["age_mean"] -
    2 * edad_summary2["age_std"]
).clip(lower=0)

edad_summary2["age_upper_2sigma"] = (
    edad_summary2["age_mean"] +
    2 * edad_summary2["age_std"]
)

display(
    edad_summary2.sort_values("age_relative_error")
)

def classify_age_uncertainty(relative_error):

    if pd.isna(relative_error):
        return "No disponible"

    elif relative_error < 0.20:
        return "Muy buena"

    elif relative_error < 0.40:
        return "Buena"

    elif relative_error < 0.60:
        return "Moderada"

    elif relative_error < 1.00:
        return "Pobre"

    else:
        return "Muy pobre"


edad_summary2["age_quality"] = (
    edad_summary2["age_relative_error"]
    .apply(classify_age_uncertainty)
)

display(
    edad_summary2[
        [
            "TIC",
            "age_mean",
            "age_std",
            "age_median",
            "age_relative_error",
            "age_quality"
        ]
    ]
)
print(
    edad_summary2["age_quality"].value_counts()
)

fig, ax = plt.subplots(figsize=(10, 7))

ax.errorbar(
    edad_summary2["age_mean"],
    np.arange(len(edad_summary2)),
    xerr=edad_summary2["age_std"],
    fmt="o",
    capsize=4
)

ax.set_yticks(np.arange(len(edad_summary2)))
ax.set_yticklabels(
    [f"TIC {tic}" for tic in edad_summary2["TIC"]]
)

ax.set_xlabel("Age", fontsize=13)
ax.set_ylabel("TIC", fontsize=13)

ax.set_title(
    "Age estimates for the Orion TIC sample somers",
    fontsize=15
)

ax.grid(
    axis="x",
    alpha=0.25,
    linestyle="--"
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Graficar distribución de edad para cada TIC
# ============================================================

for _, row in edad_summary2.iterrows():

    tic = row["TIC"]
    age_mean = row["age_mean"]
    age_std = row["age_std"]
    age_median = row["age_median"]

    # Saltar valores inválidos
    if pd.isna(age_mean) or pd.isna(age_std):
        continue

    if age_std <= 0:
        continue

    # --------------------------------------------------------
    # Rango de edades
    # --------------------------------------------------------

    age_min = max(0, age_mean - 4 * age_std)
    age_max = age_mean + 4 * age_std

    ages = np.linspace(age_min, age_max, 1000)

    # Distribución normal
    pdf = (
        1 / (age_std * np.sqrt(2 * np.pi))
        * np.exp(-0.5 * ((ages - age_mean) / age_std) ** 2)
    )

    # --------------------------------------------------------
    # Figura
    # --------------------------------------------------------

    fig, ax = plt.subplots(figsize=(8, 5))

    ax.plot(
        ages,
        pdf,
        linewidth=2
    )

    # Media
    ax.axvline(
        age_mean,
        linestyle="--",
        linewidth=1.8,
        label=f"Mean = {age_mean:.2f}"
    )

    # Mediana
    if not pd.isna(age_median):

        ax.axvline(
            age_median,
            linestyle=":",
            linewidth=2,
            label=f"Median = {age_median:.2f}"
        )

    # Región ±1 sigma
    ax.axvspan(
        age_mean - age_std,
        age_mean + age_std,
        alpha=0.15,
        label=f"$1\\sigma$ = {age_std:.2f}"
    )

    # --------------------------------------------------------
    # Labels
    # --------------------------------------------------------

    ax.set_xlabel("Age", fontsize=13)
    ax.set_ylabel("Probability density", fontsize=13)

    ax.set_title(
        f"TIC {tic}\n"
        f"Age = {age_mean:.2f} ± {age_std:.2f} Myr (Somers17)",
        fontsize=14
    )

    ax.legend()

    ax.grid(
        alpha=0.25,
        linestyle="--"
    )

    plt.tight_layout()

    # --------------------------------------------------------
    # Guardar
    # --------------------------------------------------------

    filename = output_dir / f"TIC_{tic}_age_distribution_somers17.png"

    fig.savefig(
        filename,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

# Construcción de codigo

In [ ]:
#Catálogo de resultados obtenidos de la curva de luz con tess

path_priors = "/home/luciam/Documents/starry/Orion_yceb/priors.csv"

priors = pd.read_csv(path_priors)
print(priors.columns)

In [ ]:
#Malla de tracks de mist 

path_model_1 = "/home/luciam/Documents/starry/Resultados_Orion/datos_estrellas_massage_orion/modelos/MIST_V2.csv"

mist = pd.read_csv(path_model_1)
print(mist.columns)

# Mallan de tracks Somers 17% manchas
path_model_2 = "/home/luciam/Documents/starry/Resultados_Orion/datos_estrellas_massage_orion/modelos/f017track_ms.csv"

somers17 = pd.read_csv(path_model_2)
print(somers17.columns)

In [ ]:
#Explorar encabezados de ambos catalogos
print(mist.columns)
print(somers17.columns)
print(priors.columns)


# Seleccionar TIC

In [ ]:
######################
R2_s = np.sqrt(priors['Depth_primary'])*R_median_s
R2_mist = np.sqrt(priors['Depth_primary'])*R_median_mist
########################

In [ ]:
#definir tic y explorar todos los parámetros para este objeto

# 1. Cargar tu malla de isócronas (ejemplo)
# Asegúrate de que L esté en vatios o corregido respecto a las unidades de sigma
df = pd.read_csv("mis_isocronas.csv") 

# Constante de Stefan-Boltzmann (unidades SI)
SIGMA = 5.670374e-8 

# Tu dato observado y su incertidumbre
R_obs = 1.5 * 6.957e8  # Ejemplo: 1.5 Radios Solares en metros
sigma_R = 0.1 * 6.957e8 # Error de 0.1 Radios Solares

# 2. Calcular el Radio Teórico para toda la malla de golpe (Vectorizado)
# f = sqrt( L / (T⁴ * 4 * sigma * pi) )
df['R_teo'] = np.sqrt(df['L'] / (df['T']**4 * 4 * SIGMA * np.pi))

# 3. Filtrar el conjunto de pares (L, T) válidos
# Conserva solo los puntos de las isócronas que entran en el rango de error
conjunto_valido = df[np.abs(df['R_teo'] - R_obs) <= sigma_R]

# El mejor candidato absoluto (el que minimiza el error)
mejor_par = df.loc[(np.abs(df['R_teo'] - R_obs)).idxmin()]

print(f"Se encontraron {len(conjunto_valido)} puntos compatibles en la malla.")
